# Synthetic Claims Data Generator
## Week 2 — Synthetic Data Generation
This notebook generates a synthetic healthcare claims dataset to simulate 
claim-level data with denial outcomes. The CMS dataset does not include 
denial labels — this synthetic dataset fills that gap using realistic 
denial probability rules based on known patterns in healthcare billing.

In [9]:
import pandas as pd
import numpy as np
import uuid
from faker import Faker
import random
import os

fake = Faker()
np.random.seed(42)
random.seed(42)

# Number of claims to generate
N = 50000

In [4]:
# Provider specialties
specialties = [
    'Internal Medicine',
    'Family Practice',
    'Cardiology',
    'Orthopedic Surgery',
    'Psychiatry',
    'Neurology',
    'Gastroenterology',
    'Dermatology',
    'Ophthalmology',
    'Urology',
    'Oncology',
    'Pain Management',
    'Physical Therapy',
    'Radiology',
    'Emergency Medicine'
]

# High denial specialties (used in denial logic later)
high_denial_specialties = ['Psychiatry', 'Pain Management', 'Oncology']

# Common ICD-10 codes
icd_codes = [
    'E11.9',  # Type 2 diabetes
    'I10',    # Hypertension
    'M54.5',  # Low back pain
    'J18.9',  # Pneumonia
    'F32.1',  # Major depression
    'I21.0',  # Heart attack
    'J44.1',  # COPD
    'M17.11', # Knee osteoarthritis
    'Z23',    # Immunization
    'N18.3',  # Chronic kidney disease
    'F41.1',  # Generalized anxiety
    'I50.9',  # Heart failure
    'E78.5',  # Hyperlipidemia
    'M79.3',  # Panniculitis
    'G43.909',# Migraine
    'K21.0',  # GERD
    'L40.0',  # Psoriasis
    'F20.9',  # Schizophrenia
    'C34.10', # Lung cancer
    'S72.001A'# Hip fracture
]

# CPT codes with median billed amounts
cpt_data = {
    '99213': 150,  # Office visit, moderate
    '99214': 220,  # Office visit, high
    '99232': 180,  # Subsequent hospital care
    '93000': 90,   # ECG
    '71046': 140,  # Chest X-ray
    '27447': 8000, # Knee replacement
    '80053': 60,   # Metabolic panel
    '90658': 40,   # Flu vaccine
    '99285': 600,  # Emergency visit
    '20610': 120,  # Joint injection
    '97110': 80,   # Therapeutic exercise
    '99223': 350,  # Initial hospital care
    '43239': 1200, # Upper GI endoscopy
    '70553': 1800, # Brain MRI
    '99091': 55    # Remote monitoring
}

In [5]:
def calculate_denial_probability(specialty, icd_code, cpt_code, 
                                  billed_amount, days_to_submission):
    
    # Base denial probability (realistic Medicare rate)
    denial_prob = 0.08
    
    # ICD-CPT mismatch — randomly assigned to 15% of claims
    icd_cpt_mismatch = random.random() < 0.15
    if icd_cpt_mismatch:
        denial_prob += 0.62  # brings total to 0.70
    
    # Late filing
    if days_to_submission > 60:
        denial_prob += 0.20
    
    # Upcoding signal — billed amount > 3x the CPT median
    cpt_median = cpt_data.get(cpt_code, 150)
    if billed_amount > 3 * cpt_median:
        denial_prob += 0.25
    
    # High denial specialty
    if specialty in high_denial_specialties:
        denial_prob += 0.10
    
    # Cap at 0.95 — nothing is certain
    denial_prob = min(denial_prob, 0.95)
    
    return denial_prob, icd_cpt_mismatch

In [6]:
claims = []

for _ in range(N):
    # Basic claim fields
    claim_id = str(uuid.uuid4())
    patient_age = random.randint(18, 90)
    patient_gender = random.choice(['M', 'F'])
    specialty = random.choice(specialties)
    icd_code = random.choice(icd_codes)
    cpt_code = random.choice(list(cpt_data.keys()))
    
    # Billed amount — normal distribution around CPT median
    cpt_median = cpt_data[cpt_code]
    billed_amount = max(0, np.random.normal(cpt_median, cpt_median * 0.3))
    
    # Small chance of extreme billing (upcoding simulation)
    if random.random() < 0.05:
        billed_amount = billed_amount * random.uniform(3, 6)
    
    # Days to submission
    days_to_submission = random.randint(0, 90)
    
    # Service date
    service_date = fake.date_between(start_date='-2y', end_date='today')
    
    # Calculate denial probability
    denial_prob, icd_cpt_mismatch = calculate_denial_probability(
        specialty, icd_code, cpt_code, billed_amount, days_to_submission
    )
    
    # Assign denial label
    is_denied = int(random.random() < denial_prob)
    
    claims.append({
        'claim_id': claim_id,
        'patient_age': patient_age,
        'patient_gender': patient_gender,
        'provider_specialty': specialty,
        'icd_code': icd_code,
        'cpt_code': cpt_code,
        'billed_amount': round(billed_amount, 2),
        'days_to_submission': days_to_submission,
        'service_date': service_date,
        'icd_cpt_mismatch': int(icd_cpt_mismatch),
        'is_denied': is_denied
    })

# Convert to dataframe
df_synthetic = pd.DataFrame(claims)
print(df_synthetic.shape)
print(f"Denial rate: {df_synthetic['is_denied'].mean():.2%}")
df_synthetic.head()

(50000, 11)
Denial rate: 26.46%


,claim_id,patient_age,patient_gender,provider_specialty,icd_code,cpt_code,billed_amount,days_to_submission,service_date,icd_cpt_mismatch,is_denied
0,37472b2d-fabb-4ca5-ae55-caf200ebaf18,32,M,Pain Management,Z23,93000,103.41,13,2024-06-13,0,0
1,160e91d2-09fb-473b-998a-3299ad094483,29,F,Internal Medicine,E11.9,99214,210.87,64,2026-01-25,0,0
2,1e103100-a60d-46e3-9b7f-7e068a453542,87,F,Orthopedic Surgery,G43.909,20610,143.32,0,2025-09-04,0,0
3,3b2e9feb-2baa-4b40-88b1-a4222c680bb2,72,F,Psychiatry,F32.1,93000,131.12,43,2026-02-28,1,1
4,28f595be-f94a-4ec9-b99f-754e261bc810,63,F,Urology,Z23,43239,4884.86,15,2026-03-05,0,0


In [11]:
os.makedirs('../data/synthetic', exist_ok=True)
df_synthetic.to_csv('../data/synthetic/synthetic_claims.csv', index=False)
print("Saved successfully")

Saved successfully


In [1]:
import os
print(os.getcwd())

/Users/darshanbalaji/Documents/healthcare-claims-ai/healthcare-claims-ai/notebooks
